# YOLOv11 Training Notebook
## Real-time Student Behavior Detection (FYP)

This notebook is rewritten in correct execution order. Run cells from top to bottom.

## 1. Import Required Libraries and Setup

In [ ]:
import sys, platform
print('Python:', sys.version)
print('Executable:', sys.executable)
print('Platform:', platform.platform())

Python: 3.12.6 (tags/v3.12.6:a4a2d2b, Sep  6 2024, 20:11:23) [MSC v.1940 64 bit (AMD64)]
Executable: d:\FYP\FYP CODE\.venv\Scripts\python.exe
Platform: Windows-11-10.0.26200-SP0


In [ ]:
%pip install -q ultralytics opencv-python pyyaml matplotlib pillow

In [ ]:
from pathlib import Path
import json
import yaml
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

def is_colab() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False

ON_COLAB = is_colab()

# Optional: mount Google Drive when running in Colab
if ON_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)

# In Colab, prefer Drive dataset first to avoid using stale local dataset/.
if ON_COLAB:
    dataset_candidates = [
        Path('/content/drive/MyDrive/dataset'),
        Path('/content/drive/MyDrive/datasets'),
        Path('/content/dataset'),
        Path('dataset'),
    ]
else:
    dataset_candidates = [
        Path('dataset'),
        Path('/content/dataset'),
        Path('/content/drive/MyDrive/dataset'),
        Path('/content/drive/MyDrive/datasets'),
    ]

resolved_dataset = None
for candidate in dataset_candidates:
    if (candidate / 'data.yaml').exists():
        resolved_dataset = candidate
        break

if resolved_dataset is None:
    resolved_dataset = Path('dataset')

DATASET_ROOT = resolved_dataset
PROJECT_ROOT = DATASET_ROOT.parent if DATASET_ROOT.parent != Path('') else Path('.')
DATA_YAML = DATASET_ROOT / 'data.yaml'

# Save training runs to Drive when in Colab, local folder otherwise.
RUNS_ROOT = Path('/content/drive/MyDrive/fyp_runs') if ON_COLAB else Path('fyp_runs')
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = RUNS_ROOT / 'classroom_model_v2'
BEST_WEIGHTS = RUN_DIR / 'weights' / 'best.pt'

# Device selection
device = '0' if torch.cuda.is_available() else 'cpu'
print(f'Running on Colab: {ON_COLAB}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset root: {DATASET_ROOT}')
print(f'Data yaml: {DATA_YAML}')
print(f'Runs root: {RUNS_ROOT}')
print(f'PyTorch version: {torch.__version__}')
print(f'torch.cuda.is_available(): {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device name: {torch.cuda.get_device_name(0)}')
else:
    if ON_COLAB:
        print('WARNING: Colab GPU is not active. Change Runtime -> GPU and restart runtime.')
print(f'Training device setting: {device}')

if not DATA_YAML.exists():
    print('Dataset not found. Expected path like /content/drive/MyDrive/dataset/data.yaml')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running on Colab: True
Project root: /content/drive/MyDrive
Dataset root: /content/drive/MyDrive/dataset
Data yaml: /content/drive/MyDrive/dataset/data.yaml
Runs root: /content/drive/MyDrive/fyp_runs
PyTorch version: 2.10.0+cpu
torch.cuda.is_available(): False
Training device setting: cpu


In [ ]:
from pathlib import Path

# Colab-first dataset resolution (no copying large folders).
dataset_candidates = [
    Path('/content/drive/MyDrive/dataset'),
    Path('/content/drive/MyDrive/datasets'),
    Path('/content/dataset'),
    Path('dataset'),
]

resolved = None
for candidate in dataset_candidates:
    if (candidate / 'data.yaml').exists():
        resolved = candidate
        break

if resolved is None:
    raise FileNotFoundError(
        "Could not find data.yaml. Expected one of: "
        "/content/drive/MyDrive/dataset/data.yaml or dataset/data.yaml"
    )

DATASET_ROOT = resolved
DATA_YAML = DATASET_ROOT / 'data.yaml'
print(f'Using dataset root: {DATASET_ROOT}')
print(f'Using data.yaml: {DATA_YAML}')

for split in ['train', 'valid', 'test']:
    for sub in ['images', 'labels']:
        p = DATASET_ROOT / split / sub
        print(f"{'OK ' if p.exists() else 'MISS'} - {p}")

Found Drive dataset candidate: /content/drive/MyDrive/dataset
Copied data.yaml from Drive.
Copying missing split: /content/drive/MyDrive/dataset/train -> dataset/train


## 1.5 Colab Dataset Path Check (No Copy)

This cell verifies your dataset location and sets `DATASET_ROOT` without copying files. Recommended Drive path: `/content/drive/MyDrive/dataset`.

## 2. Validate Dataset Structure

In [1]:
expected_dirs = [
    DATASET_ROOT / 'train' / 'images',
    DATASET_ROOT / 'train' / 'labels',
    DATASET_ROOT / 'valid' / 'images',
    DATASET_ROOT / 'valid' / 'labels',
    DATASET_ROOT / 'test' / 'images',
    DATASET_ROOT / 'test' / 'labels',
]

print('Dataset directory check:')
all_ok = True
for p in expected_dirs:
    ok = p.exists()
    print(f"{'OK ' if ok else 'MISS'} - {p}")
    all_ok = all_ok and ok

if not all_ok:
    raise FileNotFoundError('One or more required dataset folders are missing. Fix dataset structure first.')

print('All required dataset directories exist.')

NameError: name 'DATASET_ROOT' is not defined

## 3. Configure Dataset YAML

In [ ]:
if not DATA_YAML.exists():
    raise FileNotFoundError(f'Missing file: {DATA_YAML}')

with open(DATA_YAML, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

# Auto-normalize common Roboflow exports (../train/images -> train/images)
normalize_map = {
    '../train/images': 'train/images',
    '../valid/images': 'valid/images',
    '../test/images': 'test/images',
}
changed = False
for k in ['train', 'val', 'test']:
    cur = data_cfg.get(k)
    if cur in normalize_map:
        data_cfg[k] = normalize_map[cur]
        changed = True

if changed:
    with open(DATA_YAML, 'w', encoding='utf-8') as f:
        yaml.safe_dump(data_cfg, f, sort_keys=False)
    print('Updated data.yaml paths to local-relative format.')

print('Loaded data.yaml:')
print(json.dumps({
    'train': data_cfg.get('train'),
    'val': data_cfg.get('val'),
    'test': data_cfg.get('test'),
    'nc': data_cfg.get('nc'),
    'names': data_cfg.get('names'),
}, indent=2))

expected_paths = {
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
}
expected_names = ['handrise', 'read', 'write', 'sleep', 'using_device', 'stand', 'look_forward', 'turn_head']

for k, v in expected_paths.items():
    if data_cfg.get(k) != v:
        raise ValueError(f"data.yaml {k} should be '{v}', got '{data_cfg.get(k)}'")

names = data_cfg.get('names', [])
nc = int(data_cfg.get('nc', -1))
if nc != len(names):
    raise ValueError(f"nc ({nc}) does not match len(names) ({len(names)})")

if list(names) != expected_names:
    print('Warning: class names differ from expected master labels.')
    print('Expected:', expected_names)
    print('Found   :', names)
else:
    print('Class labels match expected master labels.')

print('data.yaml validation complete.')

Loaded data.yaml:
{
  "train": "train/images",
  "val": "valid/images",
  "test": "test/images",
  "nc": 6,
  "names": [
    "handrise",
    "read",
    "sleep",
    "stand",
    "using_electronic_devices",
    "write"
  ]
}
Class labels match expected exported labels.
data.yaml validation complete.


## 4. Train YOLOv11 Model

In [ ]:
model = YOLO('yolo11m.pt')

# Prevent accidental long CPU training in Colab.
ALLOW_COLAB_CPU_TRAINING = False
if ON_COLAB and device == 'cpu' and not ALLOW_COLAB_CPU_TRAINING:
    raise RuntimeError(
        'Colab is currently CPU-only. Go to Runtime > Change runtime type > GPU, then Runtime > Restart session, and rerun from Cell 1.'
    )

# Colab default: full run for actual training.
# Set QUICK_RUN=True only for a short pipeline test.
QUICK_RUN = False

if QUICK_RUN:
    epochs = 10
    imgsz = 512
    batch = 8
    patience = 8
else:
    epochs = 100
    imgsz = 640
    # AutoBatch tries to pick the largest safe batch size.
    # If Colab T4 throws OOM, set this manually to 16 or 8.
    batch = -1
    patience = 20

train_config = {
    'data': str(DATA_YAML),
    'epochs': epochs,
    'imgsz': imgsz,
    'batch': batch,
    'device': device,
    'project': str(RUNS_ROOT),
    'name': 'classroom_model_v2',
    'verbose': True,
    'save': True,
    'workers': 8,
    'cache': False,
    'amp': True,
    'optimizer': 'auto',
    'cos_lr': True,
    'patience': patience,
    'plots': True,
    'close_mosaic': 10,
    'seed': 42,
}

print('Training config:')
for k, v in train_config.items():
    print(f'  {k}: {v}')

try:
    train_results = model.train(**train_config)
    print('Training complete.')
except RuntimeError as e:
    msg = str(e).lower()
    if 'out of memory' in msg or 'cudnn' in msg:
        print('OOM/accelerator error detected. If needed, rerun with batch=16 or batch=8.')
    raise

print(f'Expected best weights at: {BEST_WEIGHTS}')

Training config:
  data: /content/drive/MyDrive/dataset/data.yaml
  epochs: 20
  imgsz: 640
  batch: 16
  device: 0
  project: fyp_runs
  name: classroom_model_v1
  verbose: True
  save: True
Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup

KeyboardInterrupt: 

## 5. Test the Trained Model

This cell runs evaluation on the held-out test split using the best weights saved during training.

In [ ]:
if not BEST_WEIGHTS.exists():
    raise FileNotFoundError(f'Missing trained weights: {BEST_WEIGHTS}. Run the training cell first.')

test_model = YOLO(str(BEST_WEIGHTS))

test_config = {
    'data': str(DATA_YAML),
    'split': 'test',
    'imgsz': 640,
    'batch': batch if isinstance(batch, int) and batch > 0 else 16,
    'device': device,
    'project': str(RUNS_ROOT),
    'name': 'classroom_model_v2_test',
    'verbose': True,
    'save': True,
    'plots': True,
}

print('Testing config:')
for k, v in test_config.items():
    print(f'  {k}: {v}')

test_results = test_model.val(**test_config)
print('Testing complete.')

results_dir = Path(getattr(test_results, 'save_dir', RUNS_ROOT / 'classroom_model_v2_test'))
print(f'Test artifacts saved to: {results_dir}')

if hasattr(test_results, 'results_dict'):
    print('Metrics:')
    print(json.dumps(test_results.results_dict, indent=2, default=str))

confusion_matrix_path = results_dir / 'confusion_matrix.png'
results_png_path = results_dir / 'results.png'
print(f'Confusion matrix exists: {confusion_matrix_path.exists()}')
print(f'Results plot exists: {results_png_path.exists()}')

## 6. Visualize Training Results

In [ ]:
results_png = RUN_DIR / 'results.png'
cm_png = RUN_DIR / 'confusion_matrix.png'
val_batch_png = RUN_DIR / 'val_batch0_pred.jpg'

print('Artifacts:')
for p in [results_png, cm_png, val_batch_png]:
    print(f"{'OK ' if p.exists() else 'MISS'} - {p}")

# Show charts if available
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
images = [results_png, cm_png, val_batch_png]
titles = ['Training Curves', 'Confusion Matrix', 'Sample Validation Predictions']

for ax, img_path, title in zip(axes, images, titles):
    ax.set_title(title)
    ax.axis('off')
    if img_path.exists():
        img = Image.open(img_path)
        ax.imshow(img)
    else:
        ax.text(0.5, 0.5, 'Not generated yet', ha='center', va='center')

plt.tight_layout()
plt.show()